# NJ BOE TGES — Middletown vs Peer Group: Distribution Analysis (2011–2025)

**District**: Middletown Township, Monmouth County (#3160)  
**Peer group**: Group G — K-12 districts with 3,501+ students  

Each chart shows **Middletown's value over time** against the **peer group distribution**:
- 🔵 **±1σ band** — 68% of peer districts fall here (normal range)
- 🟡 **±2σ band** — 95% of peer districts fall here
- 🔴 **±3σ band** — 99.7% of peer districts fall here
- A value outside the ±3σ band is an **extreme outlier** (< 0.3% of districts)

Hover for exact value and z-score.

In [1]:
import warnings
from pathlib import Path
import pandas as pd
import plotly.graph_objects as go
warnings.filterwarnings('ignore')

# ── Locate project root robustly ───────────────────────────────────────────
# Walk up from the current working directory (where JupyterLab was launched)
# until we find the directory that contains data/TGES/.  This works regardless
# of whether the notebook lives in notebooks/, code/notebooks/, or anywhere
# else, and regardless of where `jupyter lab` was started.
def _find_project_root() -> Path:
    candidate = Path().resolve()
    for _ in range(6):
        if (candidate / 'data' / 'TGES').is_dir():
            return candidate
        candidate = candidate.parent
    raise FileNotFoundError(
        f"Could not find project root (searched for data/TGES/ starting from "
        f"{Path().resolve()}).  Make sure you started JupyterLab from inside "
        f"the Middletown_Schools project directory."
    )

PROJECT_ROOT  = _find_project_root()
TGES_ROOT     = PROJECT_ROOT / 'data' / 'TGES'
YEARS         = list(range(2011, 2026))
PEER_GROUP    = 'G. K-12 / 3501 +'
DISTRICT_NAME = 'Middletown Twp'
NULL_VALS     = {'N.A.', 'N.R.', '', 'N/A', 'NA'}

# ── Extra districts to overlay on every chart ──────────────────────────────
# Each entry: (distname_in_csv, column_key, display_label, line_color)
EXTRA_DISTRICTS = [
    ('Rumson Boro',           'rumson_elem', 'Rumson Boro (K-8)',         '#6A4C93'),
    ('Rumson-Fair Haven Reg', 'rumson_hs',   'Rumson-Fair Haven (7-12)',  '#F4A261'),
]

# ── Helpers ────────────────────────────────────────────────────────────────
def get_csv_dir(year):
    base = TGES_ROOT / str(year) / 'extracted'
    for d in base.iterdir():
        if d.is_dir() and (d / 'CSG1.CSV').exists():
            return d
    return None

def clean_num(series):
    return pd.to_numeric(
        series.astype(str).str.strip().replace(list(NULL_VALS), None),
        errors='coerce')

def load_peer_values(csv_dir, fname, col):
    """
    Returns (peers_df, mtown_val) for one year/indicator.

    peers_df  — DataFrame with columns [DISTNAME, _v] for every peer district
                EXCEPT Middletown and group summary rows (non-numeric DIST).
                Mean and std are computed without Middletown so the district
                doesn't influence its own reference distribution.
    mtown_val — Middletown's numeric value, or None if missing.
    """
    match = next((f for f in csv_dir.iterdir()
                  if f.name.upper() == fname.upper()), None)
    if match is None:
        return pd.DataFrame(columns=['DISTNAME', '_v']), None
    df = pd.read_csv(match, encoding='latin-1', dtype=str)
    df.columns = df.columns.str.upper().str.strip()
    col = col.upper()
    if col not in df.columns:
        return pd.DataFrame(columns=['DISTNAME', '_v']), None
    grp  = df[df['GROUP'].str.strip() == PEER_GROUP].copy()
    # Real districts only (numeric DIST — excludes group average/median rows)
    real = grp[pd.to_numeric(grp['DIST'], errors='coerce').notna()].copy()
    real['_v'] = clean_num(real[col])
    is_mtown  = real['DISTNAME'].str.upper().str.strip() == DISTRICT_NAME.upper()
    mtown_row = real[is_mtown]
    mtown_val = mtown_row['_v'].iloc[0] if not mtown_row.empty else None
    # Peer distribution excludes Middletown so it doesn't influence its own stats
    peers_df  = real.loc[~is_mtown, ['DISTNAME', '_v']].dropna(subset=['_v']).copy()
    return peers_df, mtown_val

def load_district_value(csv_dir, fname, col, distname):
    """Return the numeric value for any named district from a CSG file."""
    match = next((f for f in csv_dir.iterdir()
                  if f.name.upper() == fname.upper()), None)
    if match is None:
        return None
    df = pd.read_csv(match, encoding='latin-1', dtype=str)
    df.columns = df.columns.str.upper().str.strip()
    col = col.upper()
    if col not in df.columns:
        return None
    row = df[df['DISTNAME'].str.strip() == distname]
    if row.empty:
        return None
    return clean_num(row.iloc[0:1][col]).iloc[0]

print(f'Project root: {PROJECT_ROOT}')
print(f'Peer group: "{PEER_GROUP}"  |  District: "{DISTRICT_NAME}"')
print(f'Extra overlays: {[d for d,_,_,_ in EXTRA_DISTRICTS]}')

Project root: /Users/loganbest/Desktop/Projects/Middletown_Schools
Peer group: "G. K-12 / 3501 +"  |  District: "Middletown Twp"
Extra overlays: ['Rumson Boro', 'Rumson-Fair Haven Reg']


In [2]:
# ── Build master stats table ────────────────────────────────────────────────
#
# ALL_INDICATORS: (csv_file, value_col, label, unit_fmt, y_label)
#   unit_fmt: '$' = dollars, 'ratio' = students per staff, 'salary' = annual salary

ALL_INDICATORS = [
    # Overall
    ('CSG1.CSV',  'PP11',      'Budgetary Per-Pupil Cost',       '$',      'Per-pupil cost ($)'),
    # Classroom
    ('CSG2.CSV',  'PP12',      'Classroom Instruction',          '$',      'Per-pupil cost ($)'),
    ('CSG3.CSV',  'PP13',      'Classroom Salaries & Benefits',  '$',      'Per-pupil cost ($)'),
    ('CSG4.CSV',  'PP14',      'Classroom Supplies/Textbooks',   '$',      'Per-pupil cost ($)'),
    ('CSG5.CSV',  'PP15',      'Classroom Purchased Services',   '$',      'Per-pupil cost ($)'),
    # Support
    ('CSG6.CSV',  'PP16',      'Support Services',               '$',      'Per-pupil cost ($)'),
    ('CSG7.CSV',  'PP17',      'Support Salaries & Benefits',    '$',      'Per-pupil cost ($)'),
    # Administration
    ('CSG8.CSV',  'PP18',      'Total Administration',           '$',      'Per-pupil cost ($)'),
    ('CSG8A.CSV', 'PP18A',     'Legal Services',                 '$',      'Per-pupil cost ($)'),
    ('CSG9.CSV',  'PP19',      'Admin Salaries & Benefits',      '$',      'Per-pupil cost ($)'),
    # Operations
    ('CSG10.CSV', 'PP110',     'Operations & Maintenance',       '$',      'Per-pupil cost ($)'),
    ('CSG11.CSV', 'PP111',     'O&M Salaries & Benefits',        '$',      'Per-pupil cost ($)'),
    # Other
    ('CSG13.CSV', 'PP113',     'Extracurricular Costs',          '$',      'Per-pupil cost ($)'),
    ('CSG15.CSV', 'PP115',     'Equipment Costs',                '$',      'Per-pupil cost ($)'),
    # Staffing ratios
    ('CSG16.CSV', 'STRAT0016', 'Student:Teacher Ratio',          'ratio',  'Students per teacher'),
    ('CSG17.CSV', 'SSRAT0017', 'Student:Support Staff Ratio',    'ratio',  'Students per support staff'),
    ('CSG18.CSV', 'SARAT0018', 'Student:Admin Ratio',            'ratio',  'Students per administrator'),
    # Salaries
    ('CSG16.CSV', 'SALT0016',  'Median Teacher Salary',          'salary', 'Median annual salary ($)'),
    ('CSG17.CSV', 'SALS0017',  'Median Support Staff Salary',    'salary', 'Median annual salary ($)'),
    ('CSG18.CSV', 'SALAM0018', 'Median Admin Salary',            'salary', 'Median annual salary ($)'),
]

stat_rows = []
for fname, col, label, fmt, y_label in ALL_INDICATORS:
    for year in YEARS:
        csv_dir = get_csv_dir(year)
        if csv_dir is None:
            continue
        peers_df, mtown_val = load_peer_values(csv_dir, fname, col)
        if len(peers_df) < 5:
            continue
        peer_vals = peers_df['_v']
        mean_v = peer_vals.mean()
        std_v  = peer_vals.std(ddof=1)
        zscore = (mtown_val - mean_v) / std_v if mtown_val is not None and std_v > 0 else None

        # Actual percentiles of the peer distribution (not theoretical σ bounds)
        p10, p25, p50, p75, p90 = peer_vals.quantile([0.10, 0.25, 0.50, 0.75, 0.90]).values

        # Middletown's percentile rank among peers (0–100)
        pctile_rank = (peer_vals < mtown_val).mean() * 100 if mtown_val is not None else None

        # Highest and lowest peer districts this year
        max_idx  = peer_vals.idxmax()
        min_idx  = peer_vals.idxmin()
        max_dist = peers_df.loc[max_idx, 'DISTNAME']
        min_dist = peers_df.loc[min_idx, 'DISTNAME']
        extra_vals = {
            key: load_district_value(csv_dir, fname, col, distname)
            for distname, key, _, _ in EXTRA_DISTRICTS
        }
        stat_rows.append(dict(
            label=label, fname=fname, col=col, fmt=fmt, y_label=y_label,
            year=year, n=len(peers_df),
            mean=mean_v, std=std_v,
            p10=p10, p25=p25, p50=p50, p75=p75, p90=p90,
            mtown=mtown_val, zscore=zscore, pctile_rank=pctile_rank,
            max_val=peer_vals.max(), max_dist=max_dist,
            min_val=peer_vals.min(), min_dist=min_dist,
            **extra_vals,
        ))

df_stats = pd.DataFrame(stat_rows)
print(f'Rows: {len(df_stats)}  |  Indicators: {df_stats["label"].nunique()}  |  Years: {df_stats["year"].nunique()}')
print('\n2025 z-scores:')
(
    df_stats[df_stats.year == 2025]
    [['label', 'mtown', 'mean', 'std', 'zscore', 'n']]
    .sort_values('zscore')
    .round(2)
)

Rows: 296  |  Indicators: 20  |  Years: 15

2025 z-scores:


,label,mtown,mean,std,zscore,n
103,Support Salaries & Benefits,1691.0,2572.55,639.25,-1.38,95
88,Support Services,2248.0,3053.67,716.07,-1.13,95
205,Equipment Costs,35.0,127.65,135.45,-0.68,95
117,Total Administration,1579.0,1844.48,405.67,-0.65,95
235,Student:Support Staff Ratio,62.3,69.94,16.43,-0.46,95
220,Student:Teacher Ratio,11.4,11.80,1.17,-0.34,95
14,Budgetary Per-Pupil Cost,17503.0,18360.61,2714.94,-0.32,95
146,Admin Salaries & Benefits,1437.0,1514.32,348.05,-0.22,95
160,Operations & Maintenance,2129.0,2206.43,584.12,-0.13,95
190,Extracurricular Costs,303.0,315.46,115.39,-0.11,95


In [3]:
# ── Percentile-band chart function ─────────────────────────────────────────
#
# Bands are built from ACTUAL peer percentiles, not theoretical ±σ multiples.
# This means:
#   - min/max lines always sit outside (or exactly at) the outer band edges
#   - right-skewed spending distributions are represented honestly
#   - interpretation is intuitive: "X% of districts fall in this range"
#
# Band layout (outer → inner, so each inner overwrites the outer visually):
#   p10–p90  : middle 80% of peers  (lightest)
#   p25–p75  : middle 50% of peers  (IQR, medium)

PCTILE_BANDS = [
    ('p25', 'p75', 'rgba(66,165,245,0.28)', 'Middle 50%  (p25–p75)'),
]

def fmt_val(v, fmt):
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return 'N/A'
    return f'${v:,.0f}' if fmt in ('$', 'salary') else f'{v:.2f}'


def _y_range(sub, fmt):
    """
    Compute a y-axis display range that clips extreme outliers.
    Uses Tukey's 3×IQR fence on the peer IQR columns, then adds 10% padding.
    Values outside the fence still appear in the traces but are clipped by the axis.
    The lower bound is always 0 for dollar/ratio data.
    """
    p25 = sub['p25'].max()   # use the widest IQR across all years
    p75 = sub['p75'].max()
    iqr = p75 - p25
    upper_fence = p75 + 3.0 * iqr
    # Also make sure Middletown is always visible
    mt_max = sub['mtown'].max(skipna=True)
    y_max = max(upper_fence, mt_max if pd.notna(mt_max) else 0) * 1.10
    y_min = 0  # spending and ratios are never negative
    return [y_min, y_max]


def sigma_chart(label, fmt, y_label, height=800):
    """
    Build an interactive Plotly figure for one indicator.

    Shaded bands show the actual peer percentile distribution (p10–p90 and
    p25–p75), so the min/max lines are always outside the bands.
    Median and mean reference lines are included.
    Middletown is overlaid with percentile rank and z-score in the hover.
    """
    sub = df_stats[df_stats['label'] == label].sort_values('year').copy()
    if sub.empty:
        return go.Figure().update_layout(title=f'{label} — no data')

    years = sub['year'].values
    mt    = sub['mtown'].values
    pr    = sub['pctile_rank'].values
    ns    = sub['n'].values
    maxv  = sub['max_val'].values
    minv  = sub['min_val'].values
    maxd  = sub['max_dist'].values
    mind  = sub['min_dist'].values

    fig = go.Figure()

    # ── Percentile bands: outer → inner ────────────────────────────────────
    first_band = True
    for lo_col, hi_col, color, band_name in PCTILE_BANDS:
        upper = sub[hi_col].values
        lower = sub[lo_col].values
        fig.add_trace(go.Scatter(
            x=years, y=upper, mode='lines',
            line=dict(width=0), showlegend=False, hoverinfo='skip',
        ))
        fig.add_trace(go.Scatter(
            x=years, y=lower, mode='lines',
            line=dict(width=0),
            fill='tonexty', fillcolor=color,
            name=band_name, legendgroup='bands',
            showlegend=first_band, hoverinfo='skip',
        ))
        first_band = False

    # ── Highest peer district ───────────────────────────────────────────────
    fig.add_trace(go.Scatter(
        x=years, y=maxv, mode='lines+markers',
        line=dict(color='#2A9D8F', width=1.2, dash='dash'),
        marker=dict(size=5, color='#2A9D8F', symbol='triangle-up'),
        name='Highest peer',
        hovertemplate='<b>%{x}</b><br>Highest: %{text}<extra>Highest peer</extra>',
        text=[f'{fmt_val(v, fmt)} — {d}' for v, d in zip(maxv, maxd)],
    ))

    # ── Lowest peer district ────────────────────────────────────────────────
    fig.add_trace(go.Scatter(
        x=years, y=minv, mode='lines+markers',
        line=dict(color='#F4A261', width=1.2, dash='dash'),
        marker=dict(size=5, color='#F4A261', symbol='triangle-down'),
        name='Lowest peer',
        hovertemplate='<b>%{x}</b><br>Lowest: %{text}<extra>Lowest peer</extra>',
        text=[f'{fmt_val(v, fmt)} — {d}' for v, d in zip(minv, mind)],
    ))

    # ── Peer median ─────────────────────────────────────────────────────────
    fig.add_trace(go.Scatter(
        x=years, y=sub['p50'].values, mode='lines',
        line=dict(color='#457B9D', width=1.5, dash='dot'),
        name='Peer median',
        hovertemplate='<b>%{x}</b><br>Peer median: %{text}<extra>Peer median</extra>',
        text=[fmt_val(v, fmt) for v in sub['p50'].values],
    ))

    # ── Extra overlay districts (e.g. Rumson) ──────────────────────────────
    for distname, key, disp_label, color in EXTRA_DISTRICTS:
        if key not in sub.columns:
            continue
        vals = sub[key].values
        hover = [
            f'{fmt_val(v, fmt)}' if v is not None and not pd.isna(v) else 'N/A'
            for v in vals
        ]
        fig.add_trace(go.Scatter(
            x=years, y=vals, mode='lines+markers',
            line=dict(color=color, width=2, dash='dashdot'),
            marker=dict(size=7, color=color, symbol='diamond',
                        line=dict(color='white', width=1)),
            name=disp_label,
            hovertemplate=f'<b>%{{x}}</b><br>{disp_label}: %{{text}}<extra>{disp_label}</extra>',
            text=hover,
        ))

    # ── Middletown line ─────────────────────────────────────────────────────
    mtown_hover = []
    for y, v, p, n in zip(years, mt, pr, ns):
        if v is None or pd.isna(v):
            mtown_hover.append('N/A')
            continue
        if p is not None and not pd.isna(p):
            if   p < 10:  p_desc = f'{p:.0f}th percentile  (bottom 10% of peers)'
            elif p < 25:  p_desc = f'{p:.0f}th percentile  (below peer IQR)'
            elif p <= 75: p_desc = f'{p:.0f}th percentile  (within peer IQR)'
            elif p <= 90: p_desc = f'{p:.0f}th percentile  (above peer IQR)'
            else:         p_desc = f'{p:.0f}th percentile  (top 10% of peers)'
        else:
            p_desc = ''
        mtown_hover.append(f'{fmt_val(v, fmt)}<br>{p_desc}<br>vs {n} peer districts')

    # Dot color by percentile position relative to peer IQR
    dot_colors = []
    for p in pr:
        if p is None or pd.isna(p):        dot_colors.append('#999')
        elif 25 <= p <= 75:                dot_colors.append('#E63946')   # within IQR — normal
        elif 10 <= p < 25 or 75 < p <= 90: dot_colors.append('#E97D23')  # outside IQR
        else:                              dot_colors.append('#9B1D1D')   # outside p10/p90

    # Label on every point: "Nth%" computed against that year's peer group
    point_labels = [
        f'{p:.0f}%' if (p is not None and not pd.isna(p)) else ''
        for p in pr
    ]

    fig.add_trace(go.Scatter(
        x=years, y=mt,
        mode='lines+markers+text',
        line=dict(color='#E63946', width=2.5),
        marker=dict(size=9, color=dot_colors,
                    line=dict(color='white', width=1.5)),
        text=point_labels,
        textposition='top center',
        textfont=dict(size=10, color='#333'),
        name='Middletown Twp',
        hovertemplate='<b>%{x}</b><br>Middletown: %{customdata}<extra>Middletown</extra>',
        customdata=mtown_hover,
    ))

    fig.update_layout(
        title=dict(text=label, font=dict(size=15), x=0),
        xaxis=dict(title='School Year', dtick=1, gridcolor='#eee', tickangle=-45),
        yaxis=dict(
            title=y_label, gridcolor='#eee',
            tickprefix='$' if fmt in ('$', 'salary') else '',
            tickformat=',' if fmt in ('$', 'salary') else '.1f',
            range=_y_range(sub, fmt),
        ),
        plot_bgcolor='white', paper_bgcolor='white',
        hovermode='x unified',
        legend=dict(orientation='h', yanchor='bottom', y=1.01,
                    xanchor='left', x=0, font=dict(size=11)),
        height=height,
        margin=dict(t=80, b=60, l=80, r=80),
    )
    return fig


print('sigma_chart() defined.')

sigma_chart() defined.


---
## Overall Spending

In [4]:
sigma_chart('Budgetary Per-Pupil Cost', '$', 'Per-pupil cost ($)').show()

---
## Classroom Spending

In [5]:
sigma_chart('Classroom Instruction', '$', 'Per-pupil cost ($)').show()

In [6]:
sigma_chart('Classroom Salaries & Benefits', '$', 'Per-pupil cost ($)').show()

In [7]:
sigma_chart('Classroom Supplies/Textbooks', '$', 'Per-pupil cost ($)').show()

In [8]:
sigma_chart('Classroom Purchased Services', '$', 'Per-pupil cost ($)').show()

---
## Support Services

In [9]:
sigma_chart('Support Services', '$', 'Per-pupil cost ($)').show()

In [10]:
sigma_chart('Support Salaries & Benefits', '$', 'Per-pupil cost ($)').show()

---
## Administration

In [11]:
sigma_chart('Total Administration', '$', 'Per-pupil cost ($)').show()

In [12]:
sigma_chart('Legal Services', '$', 'Per-pupil cost ($)').show()

In [13]:
sigma_chart('Admin Salaries & Benefits', '$', 'Per-pupil cost ($)').show()

---
## Operations & Maintenance

In [14]:
sigma_chart('Operations & Maintenance', '$', 'Per-pupil cost ($)').show()

In [15]:
sigma_chart('O&M Salaries & Benefits', '$', 'Per-pupil cost ($)').show()

---
## Other Costs

In [16]:
sigma_chart('Extracurricular Costs', '$', 'Per-pupil cost ($)').show()

In [17]:
sigma_chart('Equipment Costs', '$', 'Per-pupil cost ($)').show()

---
## Staffing Ratios

> Higher ratio = fewer staff per student

In [18]:
sigma_chart('Student:Teacher Ratio', 'ratio', 'Students per teacher').show()

In [19]:
sigma_chart('Student:Support Staff Ratio', 'ratio', 'Students per support staff').show()

In [20]:
sigma_chart('Student:Admin Ratio', 'ratio', 'Students per administrator').show()

---
## Staff Salaries

In [21]:
sigma_chart('Median Teacher Salary', 'salary', 'Median annual salary ($)').show()

In [22]:
sigma_chart('Median Support Staff Salary', 'salary', 'Median annual salary ($)').show()

In [23]:
sigma_chart('Median Admin Salary', 'salary', 'Median annual salary ($)').show()

---
## Summary: z-scores across all indicators (2025)

Sorted from most below average → most above average.

In [24]:
import plotly.express as px

summary = (
    df_stats[df_stats.year == 2025]
    .dropna(subset=['zscore'])
    .sort_values('zscore')
    .copy()
)
summary['rank_label'] = summary['zscore'].apply(lambda z:
    f'z = {z:+.2f}σ' + (
        '  ★ notable'  if 1 < abs(z) <= 2 else
        '  ⚠ unusual'  if 2 < abs(z) <= 3 else
        '  ⛔ outlier'  if abs(z) > 3 else ''
    )
)
summary['bar_color'] = summary['zscore'].apply(
    lambda z: '#2A9D8F' if z <= -1 else '#E76F51' if z >= 1 else '#999'
)

fig = go.Figure(go.Bar(
    x=summary['zscore'],
    y=summary['label'],
    orientation='h',
    marker_color=summary['bar_color'],
    text=summary['rank_label'],
    textposition='outside',
    textfont=dict(size=10),
    hovertemplate='<b>%{y}</b><br>z = %{x:+.2f}σ<extra></extra>',
))

for x, dash, label in [(-1, 'dash', '−1σ'), (1, 'dash', '+1σ'),
                        (-2, 'dot',  '−2σ'), (2, 'dot',  '+2σ')]:
    fig.add_vline(x=x, line_dash=dash, line_color='#bbb', line_width=1,
                  annotation_text=label, annotation_position='top',
                  annotation_font_size=10)
fig.add_vline(x=0, line_color='#333', line_width=1.5)

fig.update_layout(
    title=dict(text='Middletown z-scores across all indicators (2025 school year)', x=0),
    xaxis=dict(title='Standard deviations from peer group mean', gridcolor='#eee',
               range=[-2.5, 2.5]),
    yaxis=dict(gridcolor='#eee', autorange='reversed'),
    plot_bgcolor='white', paper_bgcolor='white',
    showlegend=False,
    height=560, margin=dict(l=230, r=130),
    uniformtext_minsize=8, uniformtext_mode='hide',
)
fig.show()